In [ ]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC

SEMENTE = 42
CLASSES = ("invalido", "rg_antigo", "rg_novo")
COLUNAS_EXCLUIDAS = (
    "classe", "arquivo", "texto_extraido", "pontuacao_palavras_chave",
    "tem_texto_rg", "tem_carteira_de_identidade",
)


In [ ]:
candidatos = [
    Path(os.environ["CSV_CLASSIFICADOR"])
    if os.environ.get("CSV_CLASSIFICADOR") else None,
    Path("analise_documentos.csv"),
    Path("analise_documentosT.csv"),
    Path("dados_privados/analise_documentos.csv"),
    Path("dados_privados/analise_documentosT.csv"),
    Path("/content/analise_documentos.csv"),
    Path("/content/analise_documentosT.csv"),
]
arquivo_csv = next((p for p in candidatos if p is not None and p.is_file()), None)
if arquivo_csv is None:
    raise FileNotFoundError(
        "CSV não encontrado. Faça upload de analise_documentos.csv ou "
        "analise_documentosT.csv na pasta atual do Colab/VS Code, ou "
        "configure CSV_CLASSIFICADOR. O arquivo é privado e não deve ir para o GitHub."
    )

dados = pd.read_csv(arquivo_csv, encoding="utf-8-sig")
colunas_faltantes = set(COLUNAS_EXCLUIDAS) - set(dados.columns)
if colunas_faltantes:
    raise ValueError(f"Colunas obrigatórias ausentes: {sorted(colunas_faltantes)}")
if dados.empty or dados["classe"].isna().any():
    raise ValueError("CSV vazio ou com classes ausentes.")
if dados["classe"].nunique() < 2:
    raise ValueError("É necessário ter pelo menos duas classes para treinar.")
print(f"CSV localizado: {arquivo_csv} | total de documentos: {len(dados)}")
print("Classes:\n", dados["classe"].value_counts().to_string())


In [ ]:
x = dados.drop(columns=list(COLUNAS_EXCLUIDAS))
y = dados["classe"].astype(str)
colunas_categoricas = x.select_dtypes(include=["object", "string", "category"]).columns.tolist()
colunas_numericas = [col for col in x.columns if col not in colunas_categoricas]
for coluna in colunas_numericas:
    x[coluna] = pd.to_numeric(x[coluna], errors="coerce")
if not colunas_numericas and not colunas_categoricas:
    raise ValueError("Nenhuma característica disponível para treinamento.")
if y.value_counts().min() < 2:
    raise ValueError("Cada classe precisa de ao menos duas imagens.")

x_treino, x_teste, y_treino, y_teste = train_test_split(
    x, y, test_size=0.2, random_state=SEMENTE, stratify=y
)
print(f"Treinamento: {len(x_treino)} | teste: {len(x_teste)}")


In [ ]:
def criar_preprocessador(padronizar):
    transformacoes = []
    if colunas_numericas:
        etapas_numericas = [("preencher", SimpleImputer(strategy="median"))]
        if padronizar:
            etapas_numericas.append(("padronizar", StandardScaler()))
        transformacoes.append(("numericos", Pipeline(etapas_numericas), colunas_numericas))
    if colunas_categoricas:
        transformacoes.append((
            "categoricos",
            Pipeline([
                ("preencher", SimpleImputer(strategy="most_frequent")),
                ("codificar", OneHotEncoder(handle_unknown="ignore")),
            ]),
            colunas_categoricas,
        ))
    return ColumnTransformer(transformacoes)


In [ ]:
modelos = {
    "KNN": Pipeline([
        ("preprocessar", criar_preprocessador(padronizar=True)),
        ("classificador", KNeighborsClassifier(n_neighbors=5)),
    ]),
    "SVM": Pipeline([
        ("preprocessar", criar_preprocessador(padronizar=True)),
        ("classificador", SVC(kernel="rbf", C=1, class_weight="balanced")),
    ]),
    "Random Forest": Pipeline([
        ("preprocessar", criar_preprocessador(padronizar=False)),
        ("classificador", RandomForestClassifier(n_estimators=100, random_state=SEMENTE)),
    ]),
}

resultados = []
previsoes = {}
for nome, modelo in modelos.items():
    modelo.fit(x_treino, y_treino)
    pred = modelo.predict(x_teste)
    previsoes[nome] = pred
    resultados.append({
        "Modelo": nome,
        "Acurácia": accuracy_score(y_teste, pred),
        "Precisão macro": precision_score(y_teste, pred, average="macro", zero_division=0),
        "Recall macro": recall_score(y_teste, pred, average="macro", zero_division=0),
        "F1 macro": f1_score(y_teste, pred, average="macro", zero_division=0),
    })
    print(f"\n{nome}\n" + classification_report(y_teste, pred, zero_division=0))

comparacao = pd.DataFrame(resultados).sort_values(
    by=["F1 macro", "Acurácia"], ascending=False
).reset_index(drop=True)
print("\nComparação dos modelos:\n", comparacao.round(4).to_string(index=False))


In [ ]:
for nome, pred in previsoes.items():
    ConfusionMatrixDisplay.from_predictions(y_teste, pred, labels=sorted(y.unique()),
                                            xticks_rotation=25, cmap="Blues")
    plt.title(f"Matriz de confusão — {nome}")
    plt.tight_layout()
    plt.show()

print("\nAtenção: as métricas são de um único conjunto de teste (20%). "
      "Dados de imagens relacionadas/aumentadas podem inflar a estimativa.")
